In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
import numpy as np
import datetime as dt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
warnings.filterwarnings("ignore")

rentals_df = pd.read_csv('data/rentals.csv')
off_plan_df = pd.read_csv('data/off_plan.csv')
metro_stations_df = pd.read_csv('data/metro_stations.csv')
area_monthly = pd.read_csv('data/area_prices_monthly.csv')

rentals_df.head()

In [ ]:
off_plan_df.head()

In [ ]:
metro_stations_df.head()

In [ ]:
def summarize_frame(df):
    summary = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': df.isna().mean()*100,
        'unique': df.nunique(dropna=True),
        })
    return summary.sort_values(['missing_pct', 'unique'], ascending=[False, False])

rentals_df['date_listed'] = pd.to_datetime(rentals_df['date_listed'])
date_min = rentals_df['date_listed'].min().date()
date_max = rentals_df['date_listed'].max().date()

print(f'Date range: {date_min} to {date_max}')
print(f'Rows: {len(rentals_df):,}')
display(summarize_frame(rentals_df))

target_summary = rentals_df['annual_rent_usd'].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
display(target_summary.to_frame('annual_rent_usd'))

In [ ]:
metro_stations_df.nunique()

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,4))

sns.histplot(rentals_df['annual_rent_usd'], bins=60, ax=axes[0])
axes[0].set_title('Annual rent distribution')
axes[0].set_xlabel('Annual rent, USD')

sns.histplot(np.log1p(rentals_df['annual_rent_usd']), bins=60, ax=axes[1], color='tab:green')
axes[1].set_title('Log annual rent distribution')
axes[1].set_xlabel('log1p(annual_rent_usd)')

plt.tight_layout()
plt.show()

In [ ]:
rentals_eda = rentals_df.copy()
rentals_eda['year_month'] = rentals_eda['date_listed'].dt.to_period('M').astype(str)

monthly_rent = (
    rentals_eda.groupby('year_month')
    .agg(
        median_rent=('annual_rent_usd', 'median'),
        median_rent_per_sqft=('rent_per_sqft_usd', 'median'),
        listings=('id', 'count'),
    )
    .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(14,7), sharex=True)

sns.lineplot(data=monthly_rent, x='year_month', y='median_rent', marker='o', ax=axes[0])
axes[0].set_title('Median annual rent by month')
axes[0].set_ylabel('USD')

sns.barplot(data=monthly_rent, x='year_month', y='listings', ax=axes[1])
axes[1].set_title('Rental listing count by month')
axes[1].set_ylabel('Listings')
axes[1].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
community_stats = (
    rentals_eda.groupby(['community', 'zone'])
    .agg(
        listings=('id', 'count'),
        median_rent=('annual_rent_usd', 'median'),
        median_rent_per_sqft=('rent_per_sqft_usd', 'median'),
        median_area=('area_sqft', 'median'),
    )
    .query('listings >= 80')
    .sort_values('median_rent', ascending=False)
)

display(community_stats.head(11))
display(community_stats.tail(11))

fig, axes = plt.subplots(1, 2, figsize=(15,6))

top_communities = community_stats.head(12).reset_index()
sns.barplot(data=top_communities, y='community', x='median_rent', ax=axes[0])
axes[0].set_title('Highest median annual rent')
axes[0].set_xlabel('USD')
axes[0].set_ylabel('')


property_order = rentals_eda.groupby('property_type')['annual_rent_usd'].median().sort_index().index
sns.boxplot(
    data=rentals_eda,
    y='property_type',
    x='annual_rent_usd',
    order=property_order,
    showfliers=False,
    ax=axes[1],
)

axes[1].set_title('Rent by property type')
axes[1].set_xlabel('Annual rent, USD')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
rentals_df.info()

In [ ]:
LEAKAGE_COLS = ['annual_rent_usd', 'rent_per_sqft_usd', 'rent_per_m2_usd', 'id', 'area_m2']

rentals_df['date_listed'] = pd.to_datetime(rentals_df['date_listed'])

def add_market_context(df, monthly_df):
    """Add community-month market signals available at listing time."""
    out = df.copy()
    monthly = monthly_df.copy()
    monthly["year_month"] = pd.to_datetime(monthly["year_month"]).dt.to_period("M").astype(str)

    market_cols = [
        "year_month",
        "community",
        "secondary_price_per_sqft_usd",
        "rental_price_per_sqft_annual_usd",
        "n_listings_secondary",
        "n_listings_rental",
        "cbuae_base_rate_pct",
        "avg_mortgage_rate_pct",
    ]
    monthly = monthly[market_cols].rename(columns={
        "secondary_price_per_sqft_usd": "market_secondary_ppsf",
        "rental_price_per_sqft_annual_usd": "market_rent_ppsf",
        "n_listings_secondary": "market_secondary_listings",
        "n_listings_rental": "market_rental_listings",
    })

    out["year_month"] = out["date_listed"].dt.to_period("M").astype(str)
    out = out.merge(monthly, on=["year_month", "community"], how="left")
    return out


def engineer_features(df, monthly_df):
    out = add_market_context(df, monthly_df)

    out["listing_year"] = out["date_listed"].dt.year
    out["listing_month"] = out["date_listed"].dt.month
    out["listing_quarter"] = out["date_listed"].dt.quarter
    out["days_since_start"] = (out["date_listed"] - out["date_listed"].min()).dt.days

    out["log_area_sqft"] = np.log1p(out["area_sqft"])
    out["sqrt_area_sqft"] = np.sqrt(out["area_sqft"])
    out["bedrooms_per_1000_sqft"] = out["bedrooms"] / (out["area_sqft"] / 1000).replace(0, np.nan)
    out["parking_per_bedroom"] = out["parking_spaces"] / out["bedrooms"].replace(0, 1)

    out["is_studio"] = (out["bedrooms"] == 0).astype(int)
    out["is_villa"] = (out["property_category"] == "villa").astype(int)
    out["is_furnished"] = out["furnishing"].isin(["fully_furnished", "semi_furnished"]).astype(int)
    out["is_short_term"] = (out["contract_type"] == "short_term").astype(int)

    out["metro_distance_log1p"] = np.log1p(out["metro_distance_min"])
    out["near_metro_15min"] = (out["metro_distance_min"] <= 15).astype(int)
    out["near_metro_30min"] = (out["metro_distance_min"] <= 30).astype(int)
    out["burj_distance_log1p"] = np.log1p(out["to_burj_khalifa_km"])

    out["lat_round_2"] = out["lat"].round(2).astype(str)
    out["lon_round_2"] = out["lon"].round(2).astype(str)
    out["geo_cell"] = out["lat_round_2"] + "_" + out["lon_round_2"]

    out["property_bedroom_type"] = out["property_category"] + "_" + out["bedrooms"].astype(str)
    out["community_property_type"] = out["community"] + "_" + out["property_type"]

    return out

final_df = engineer_features(rentals_df, area_monthly)
print(final_df.shape)
final_df.head()

In [ ]:
final_df = final_df.sort_values('date_listed').reset_index(drop=True)

split_idx = int(len(final_df) * 0.8)
train_df = final_df.iloc[:split_idx].copy()
test_df = final_df.iloc[split_idx:].copy()

feature_df = final_df.drop(columns=LEAKAGE_COLS + ["date_listed"], errors="ignore")
feature_columns = feature_df.columns.tolist()

X_train = train_df[feature_columns]
X_test = test_df[feature_columns]
y_train = np.log1p(train_df["annual_rent_usd"])
y_test = np.log1p(test_df["annual_rent_usd"])
y_test_usd = test_df["annual_rent_usd"].values

print("Feature count:", len(feature_columns))

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

cat_cols = X_train.select_dtypes(include="object").columns.tolist()
print("Категориальные колонки:", cat_cols)


encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train_enc = X_train.copy()
X_test_enc = X_test.copy()

X_train_enc[cat_cols] = encoder.fit_transform(X_train[cat_cols])
X_test_enc[cat_cols] = encoder.transform(X_test[cat_cols])

print(X_train_enc.dtypes.value_counts())

In [ ]:
try:
    from sklearn.metrics import root_mean_squared_error as _rmse
except Exception:
    def _rmse(y_true, y_pred):
        return np.sqrt(mean_squared_error(y_true, y_pred))

In [ ]:
def regression_metrics(y_true_usd, y_pred_usd):
    y_pred_usd = np.maximum(y_pred_usd, 0)
    rmse = _rmse(y_true_usd, y_pred_usd)
    mae = mean_absolute_error(y_true_usd, y_pred_usd)
    rmsle = _rmse(np.log1p(y_true_usd), np.log1p(y_pred_usd))
    r2 = r2_score(y_true_usd, y_pred_usd)
    mape = np.mean(np.abs((y_true_usd - y_pred_usd) / np.maximum(y_true_usd, 1))) * 100
    return {
        "RMSE_USD": rmse,
        "MAE_USD": mae,
        "RMSLE": rmsle,
        "R2": r2,
        "MAPE_pct": mape,
    }

In [ ]:
import xgboost as xgb
import lightgbm as lgb
import optuna
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
def xgb_objective(trial):
    params = {
        "n_estimators":       trial.suggest_int("n_estimators", 300, 1500),
        "max_depth":          trial.suggest_int("max_depth", 3, 10),
        "learning_rate":      trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":          trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":   trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight":   trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha":          trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":         trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
    }
    model = xgb.XGBRegressor(**params)
    scores = cross_val_score(
        model, X_train_enc, y_train,
        cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1
    )
    return scores.mean()

xgb_study = optuna.create_study(direction="maximize")
xgb_study.optimize(xgb_objective, n_trials=35, show_progress_bar=True)

print("XGB best RMSLE (log-scale):", -xgb_study.best_value)
print("XGB best params:", xgb_study.best_params)

xgb_model = xgb.XGBRegressor(**xgb_study.best_params, random_state=42, n_jobs=-1)
xgb_model.fit(X_train_enc, y_train)

xgb_pred_log = xgb_model.predict(X_test_enc)
xgb_pred_usd = np.expm1(xgb_pred_log)
xgb_metrics = regression_metrics(y_test_usd, xgb_pred_usd)
print("XGBoost metrics:", xgb_metrics)

In [ ]:
def lgb_objective(trial):
    params = {
        "n_estimators":       trial.suggest_int("n_estimators", 300, 1500),
        "max_depth":          trial.suggest_int("max_depth", 3, 10),
        "learning_rate":      trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves":         trial.suggest_int("num_leaves", 20, 300),
        "subsample":          trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":   trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_samples":  trial.suggest_int("min_child_samples", 5, 100),
        "reg_alpha":          trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":         trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }
    model = lgb.LGBMRegressor(**params)
    scores = cross_val_score(
        model, X_train_enc, y_train,
        cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1
    )
    return scores.mean()


lgb_study = optuna.create_study(direction="maximize")
lgb_study.optimize(lgb_objective, n_trials=35, show_progress_bar=True)

print("LGB best RMSLE (log-scale):", -lgb_study.best_value)
print("LGB best params:", lgb_study.best_params)

lgb_model = lgb.LGBMRegressor(**lgb_study.best_params, random_state=42, n_jobs=-1, verbose=-1)
lgb_model.fit(X_train_enc, y_train)

lgb_pred_log = lgb_model.predict(X_test_enc)
lgb_pred_usd = np.expm1(lgb_pred_log)
lgb_metrics = regression_metrics(y_test_usd, lgb_pred_usd)
print("LightGBM metrics:", lgb_metrics)

In [ ]:
results = pd.DataFrame({
    "XGBoost":  xgb_metrics,
    "LightGBM": lgb_metrics,
}).T

print(results)

In [ ]:
import shap

sample = shap.sample(X_train_enc, 1000, random_state=42)

xgb_explainer = shap.TreeExplainer(xgb_model)
xgb_shap_values = xgb_explainer.shap_values(sample)

print("XGBoost SHAP")
plt.figure()
shap.summary_plot(xgb_shap_values, X_train_enc[:1000], max_display=20, show=False)
plt.title("XGBoost — SHAP Summary")
plt.tight_layout()
plt.show()

In [ ]:
sample = shap.sample(X_train_enc, 1000, random_state=42)

lgb_explainer = shap.TreeExplainer(lgb_model)
lgb_shap_values = lgb_explainer.shap_values(sample)

print("LightGBM SHAP")
plt.figure()
shap.summary_plot(lgb_shap_values, X_train_enc[:1000], max_display=20, show=False)
plt.title("LightGBM — SHAP Summary")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

plt.sca(axes[0])
shap.summary_plot(
    xgb_shap_values, X_train_enc,
    plot_type="bar",
    max_display=20,
    show=False
)
axes[0].set_title("XGBoost — Feature Importance (SHAP)", fontsize=14, pad=15)
axes[0].set_xlabel("mean(|SHAP value|)", fontsize=11)
axes[0].tick_params(axis='y', labelsize=10)

plt.sca(axes[1])
shap.summary_plot(
    lgb_shap_values, X_train_enc,
    plot_type="bar",
    max_display=20,
    show=False
)
axes[1].set_title("LightGBM — Feature Importance (SHAP)", fontsize=14, pad=15)
axes[1].set_xlabel("mean(|SHAP value|)", fontsize=11)
axes[1].tick_params(axis='y', labelsize=10)

plt.tight_layout(pad=3.0)
plt.show()

In [ ]:
xgb_importance = pd.DataFrame({
    "feature": X_train_enc.columns,
    "shap_importance_xgb": np.abs(xgb_shap_values).mean(axis=0)
})

lgb_importance = pd.DataFrame({
    "feature": X_train_enc.columns,
    "shap_importance_lgb": np.abs(lgb_shap_values).mean(axis=0)
})

importance_df = (
    xgb_importance
    .merge(lgb_importance, on="feature")
    .assign(mean_importance=lambda x: (x["shap_importance_xgb"] + x["shap_importance_lgb"]) / 2)
    .sort_values("mean_importance", ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(10))